# Part B — Traditional Text Classification

Four traditional classifiers on the **AG News Topic Classification** dataset (4 classes, 120k train / 7.6k test, balanced):

| # | Classifier         | Features                |
|---|--------------------|-------------------------|
| 1 | Multinomial NB     | tf-idf word uni-grams   |
| 2 | Multinomial NB     | tf-idf char tri-grams   |
| 3 | LinearSVC (C=1)    | tf-idf word uni-grams   |
| 4 | LinearSVC (C=1)    | tf-idf char tri-grams   |

All texts are lowercased and built as `Title + ' ' + Description`. The dataset is downloaded once via `kagglehub` and cached under `~/.cache/kagglehub/`.

## Setup

Add `part_b_traditional_txt_classification/` to `sys.path` so the script modules import cleanly when the notebook is run from `notebooks/`.

In [1]:
import os, sys
PART_B = os.path.abspath('../part_b_traditional_txt_classification')
if PART_B not in sys.path:
    sys.path.insert(0, PART_B)

from data_utils import load_ag_news

print('Loading AG News (cached after first download) ...')
ds = load_ag_news()
print(f'  train: {len(ds.X_train):>6}  test: {len(ds.X_test):>6}')
print(f'  labels: {ds.label_names}')

/opt/homebrew/Caskroom/miniconda/base/envs/nlp-ncsr-task2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading AG News (cached after first download) ...
  train: 120000  test:   7600
  labels: ['World', 'Sports', 'Business', 'Sci/Tech']


## B.1 — Train + evaluate the four models

For each (vectorizer, classifier) pair we measure:

- **Test accuracy** (sklearn `accuracy_score`).
- **Dimensionality** = `len(vectorizer.vocabulary_)` after `fit_transform` on training data.
- **Training time (s)** = wall-clock seconds for `vectorizer.fit_transform(train) + classifier.fit(X, y)`. (Vectorization is part of training because the vocabulary itself is learned from the training corpus.)

The trained models are returned by `train_all()` so the error-analysis section can re-use them without re-training.

In [2]:
from b1_train_models import train_all, print_table

results = train_all(ds)
print()
print_table(results)

  training: NB  (word 1-grams) ...
  training: NB  (char 3-grams) ...
  training: SVM (word 1-grams) ...
  training: SVM (char 3-grams) ...

------------------------------------------------------------------------------------------------------------
metric           |   NB  (word 1-grams) |   NB  (char 3-grams) |   SVM (word 1-grams) |   SVM (char 3-grams)
------------------------------------------------------------------------------------------------------------
Accuracy         |               0.9022 |               0.8687 |               0.9196 |               0.9121
Dimensionality   |               64,999 |               31,074 |               64,999 |               31,074
Time (sec)       |                 0.96 |                 4.51 |                 3.60 |                14.11
------------------------------------------------------------------------------------------------------------


**Observations.**
- **SVM > NB** for both feature types — the larger-margin discriminative classifier consistently beats the generative independence-assumption baseline by ~2 points.
- **Word uni-grams > char tri-grams** for both classifiers — news headlines + descriptions are short and content-word-rich, which favours the word view. Char tri-grams help most when texts are misspelled, multilingual, or morphologically rich; AG News is none of these.
- **Best model**: SVM (word uni-grams) at ≈ 0.9196.
- **Char tri-grams have a smaller vocab** (~31k) than word uni-grams (~65k), but **char-feature SVM is the slowest to train** (~14 s vs ~3 s for word-SVM). Char tf-idf vectors are denser per document (every 3-character window is a feature, so a 100-character text has ~98 nonzero entries) — `LinearSVC`'s coordinate-descent optimizer pays for that density.
- NB is roughly **3× faster** than SVM for the same features, with an accuracy gap of ~2 points. Reasonable speed/quality trade-off for production-style baselines.

## B.2 — Documents misclassified by *all four* models

For each test document, we check whether **every** model predicted incorrectly. These are the "hard cases" — likely either (a) genuinely ambiguous between two classes, or (b) label-noise in the dataset. We:

1. Count the unanimously-wrong docs in total.
2. Break them down by **true** category (which class is hardest to recover from the unanimous-error set).
3. Show one such document with its true label and each model's wrong prediction.

In [3]:
from collections import Counter

n_test = len(ds.y_test)
all_wrong = [
    i for i in range(n_test)
    if all(r.y_pred_test[i] != ds.y_test[i] for r in results)
]

print(f'Test docs misclassified by ALL 4 models: {len(all_wrong)} / {n_test}'
      f' ({100*len(all_wrong)/n_test:.2f}%)')

counts = Counter(ds.y_test[i] for i in all_wrong)
print('\nPer-category counts of unanimously-wrong docs:')
for c, name in enumerate(ds.label_names):
    print(f'  {name:<10} : {counts.get(c, 0)}')

if all_wrong:
    idx = all_wrong[0]
    print('\n--- example unanimously misclassified document ---')
    print(f'  test index : {idx}')
    print(f'  true label : {ds.label_names[ds.y_test[idx]]}')
    for r in results:
        print(f'  {r.name:<22} predicted -> {ds.label_names[r.y_pred_test[idx]]}')
    print(f'\n  text:\n    {ds.X_test[idx]}')

Test docs misclassified by ALL 4 models: 341 / 7600 (4.49%)

Per-category counts of unanimously-wrong docs:
  World      : 112
  Sports     : 9
  Business   : 135
  Sci/Tech   : 85

--- example unanimously misclassified document ---
  test index : 24
  true label : Sci/Tech
  NB  (word 1-grams)     predicted -> Business
  NB  (char 3-grams)     predicted -> Business
  SVM (word 1-grams)     predicted -> Business
  SVM (char 3-grams)     predicted -> Business

  text:
    rivals try to turn tables on charles schwab by michael liedtke     san francisco (ap) -- with its low prices and iconoclastic attitude, discount stock broker charles schwab corp. (sch) represented an annoying stone in wall street's wing-tipped shoes for decades...


**Observations.**
- Roughly **4–5% of the test set** is misclassified by *every* model — a meaningful but small slice. This is a useful upper bound on what better feature engineering or stronger classifiers can recover from this representation.
- **Sports** has by far the fewest unanimous errors (single digits) — its vocabulary (`game, team, coach, season, league, ...`) is so distinctive that even the weakest of the four models gets it right.
- **Business** has the most unanimous errors. Inspecting examples shows the recurring pattern: stories about *publicly-traded tech companies, financial markets for tech stocks, or stockbrokers in the IT sector* live on the **Business / Sci-Tech boundary**, and the gold label often disagrees with the obvious reading. The Charles Schwab example is illustrative — a discount-broker story tagged `Sci/Tech` that all four models call `Business` (which is the more defensible reading).
- Most unanimous errors look more like **dataset-label noise + boundary cases** than systematic model failure. The fact that all four models — including two architecturally different families (NB vs SVM) and two very different feature views (word vs char) — agree on the wrong answer is a strong signal that the *label* is what's unusual.